# slm-learning Stage 2.5 — Colab A100 sweep

Trains every config in `experiments/configs/colab/` against the matching base model, runs the eval matrix, and pushes each adapter to Hugging Face Hub under `scrubster/dr-stein-<config-name>`.

## Prereqs (one-time)

1. **Runtime → Change runtime type → A100 GPU** (Colab Pro biases toward A100; if you get T4/V100 it'll still work, just slower).
2. **Add a Colab Secret** named `HF_TOKEN` with a Hugging Face write token (left sidebar key icon → Add new secret → toggle "Notebook access" on). Get the token from https://huggingface.co/settings/tokens.

Then: **Runtime → Run all**. Walk away.

Wall time estimate (A100): ~25-35 min for all 3 configs. On T4: ~90 min.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Clone repo (or pull if rerunning in same session)
import os
if not os.path.exists('/content/gad_slms'):
    !git clone https://github.com/MagicbornStudios/gad_slms.git /content/gad_slms
else:
    !cd /content/gad_slms && git pull --ff-only
%cd /content/gad_slms

In [ ]:
# 3. Install training dependencies. Colab already has torch+CUDA — only install what's missing.
!pip install -q transformers==4.45.2 peft trl datasets accelerate huggingface_hub bitsandbytes pyyaml

In [ ]:
# 4. HF auth from Colab Secret
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import HfApi
user = HfApi().whoami()['name']
print(f'HF user: {user}')
assert user == 'scrubster', f'expected scrubster, got {user}'

In [ ]:
# 5. Re-pull the 5k OpenMathInstruct slice (gitignored).
import json
from pathlib import Path
from datasets import load_dataset

out = Path('data/openmathinstruct_5k.jsonl')
if out.exists() and out.stat().st_size > 1_000_000:
    print(f'{out} already present')
else:
    print('streaming OpenMathInstruct-2 ...')
    ds = load_dataset('nvidia/OpenMathInstruct-2', split='train', streaming=True)
    written = 0
    with out.open('w', encoding='utf-8') as f:
        for ex in ds:
            instr = ex.get('problem') or ex.get('question')
            resp = ex.get('generated_solution')
            if not instr or not resp or len(resp) > 2500:
                continue
            f.write(json.dumps({'instruction': instr, 'command': resp}, ensure_ascii=False) + '\n')
            written += 1
            if written >= 5000:
                break
    print(f'wrote {written} pairs to {out}')

In [ ]:
# 6. Run the sweep. Each config trains, evals, pushes to HF Hub. Skips configs whose MANIFEST.json already exists (so re-running picks up new work).
import os
os.environ['PYTHONPATH'] = 'src'
os.environ['PYTHONUTF8'] = '1'
!python scripts/sweep_finetune.py --configs-dir experiments/configs/colab/

In [ ]:
# 7. Print results table
import json
from pathlib import Path

rows = []
for d in sorted(Path('experiments/runs').iterdir()):
    if not d.is_dir():
        continue
    mp = d / 'MANIFEST.json'
    if not mp.exists():
        continue
    m = json.loads(mp.read_text())
    if not m.get('config', {}).get('name', '').startswith('colab_'):
        continue
    e = m.get('eval_results', {})
    g = e.get('gad_tools', {})
    h = e.get('humaneval', {})
    s = e.get('gsm8k', {})
    rows.append({
        'name': m['config']['name'],
        'base': m['config']['base_model'].split('/')[-1],
        'train_s': m.get('elapsed_train_sec'),
        'gad': f"{g.get('passed','-')}/{g.get('total','-')}" if 'passed' in g else 'n/a',
        'humaneval': f"{h.get('passed','-')}/{h.get('total','-')}" if 'passed' in h else 'n/a',
        'gsm8k': f"{s.get('passed','-')}/{s.get('total','-')}" if 'passed' in s else 'n/a',
        'hub': m.get('hub_url', 'not pushed'),
    })

print(f"{'name':40} {'base':24} {'train':>7}  {'gad':>7}  {'humaneval':>10}  {'gsm8k':>7}")
print('-' * 110)
for r in rows:
    train_s = f"{r['train_s']:.0f}s" if r['train_s'] else '-'
    print(f"{r['name']:40} {r['base']:24} {train_s:>7}  {r['gad']:>7}  {r['humaneval']:>10}  {r['gsm8k']:>7}")
print()
for r in rows:
    print(f"  {r['name']} -> {r['hub']}")

## What's next

- **All adapters auto-published to HF Hub** under `scrubster/dr-stein-<config-name>`. Pull them locally for inference.
- **MANIFEST.json** for each run captures full config + eval scores + hub URL. Inspect via `cat experiments/runs/<name>/MANIFEST.json`.
- To run a different config set, drop new YAMLs into `experiments/configs/colab/` (or any dir) and change the `--configs-dir` flag in cell 6.
- To re-run an existing config (different hyperparams, fresh seed), pass `--rerun` to `sweep_finetune.py` in cell 6.

## Common runtime issues

- **"HF_TOKEN missing"**: add the secret per the prereqs at top, then re-run cell 4.
- **"Disconnected"**: Colab Pro lets you reconnect. Re-running the notebook is idempotent — completed configs (have `MANIFEST.json`) skip on the second pass.
- **"OOM" on T4 (16GB)**: T4 won't fit the 3B configs. Drop `qwen3b_*.yaml` files to `_disabled/` or override `--configs` to skip them.
- **"Adapter not pushed"**: cell 7 will say `not pushed`. Likely token issue. Check `hf auth whoami` in a fresh cell and re-auth via `os.environ['HF_TOKEN']`.